<a href="https://colab.research.google.com/github/goseul17/-/blob/main/Rule__Score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================
# 1. 라이브러리 설치 (코랩 처음 실행 시)
# =====================================================

!pip install pymupdf
!pip install kiwipiepy
!pip install pytesseract
!pip install python-docx
!pip install openpyxl
!pip install python-pptx

!apt-get install tesseract-ocr -y
!apt-get install tesseract-ocr-kor -y

  Using cached kiwipiepy-0.23.1-cp39-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.3 kB)
  Using cached kiwipiepy_model-0.23.0.tar.gz (88.0 MB)
  Preparing metadata (setup.py) ... done
Using cached kiwipiepy-0.23.1-cp39-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (6.3 MB)
  Created wheel for kiwipiepy_model: filename=kiwipiepy_model-0.23.0-py3-none-any.whl size=88067826 sha256=0c241c2ebb8df1c7c8aa411dfbdd349d0bd4ba82a50f2258dd6d40530cee5c3e
  Stored in directory: /root/.cache/pip/wheels/f2/94/da/ff88b4c2305cd1f3effc8001b5f42f16dc9931bcd84d1e77c3
Successfully built kiwipiepy_model
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr-kor is already the newest version (1:4.00~git30-7274c

In [ ]:
# =====================================================
# 2. 라이브러리 import
# =====================================================

import os
import zipfile
import fitz
import pytesseract
import pandas as pd


from PIL import Image

from docx import Document

from pptx import Presentation

from kiwipiepy import Kiwi

In [ ]:
# =====================================================
# 3. ZIP 압축 해제
# =====================================================

import zipfile
import os

zip_path = "/content/데이터1.zip"

extract_path = "/content/data"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:

    zip_ref.extractall(extract_path)

print("압축 해제 완료")

압축 해제 완료


In [ ]:
import zipfile
import os

zip_path = "/content/데이터2.zip"
extract_path = "/content/data"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("압축 해제 완료")

압축 해제 완료


In [ ]:
# =====================================================
# 3. Kiwi 초기화
# =====================================================

kiwi = Kiwi()

In [ ]:
# =====================================================
# 4. PDF 텍스트 추출
# =====================================================

def extract_pdf_text(path):

    doc = fitz.open(path)

    text = ""

    for page in doc[:3]:

        text += page.get_text()

        if len(text) > 5000:
            break

    return text

In [ ]:
# =====================================================
# 5. 이미지 OCR
# =====================================================

def extract_ocr(image_path):

    img = Image.open(image_path)

    text = pytesseract.image_to_string(
        img,
        lang='kor'
    )

    return text

In [ ]:
# =====================================================
# 6. DOCX 텍스트 추출
# =====================================================

def extract_docx_text(path):

    doc = Document(path)

    text = ""

    for para in doc.paragraphs:

        text += para.text + "\n"

    return text

In [ ]:
# =====================================================
# 7. XLSX 텍스트 추출
# =====================================================

def extract_xlsx_text(path):

    excel_file = pd.ExcelFile(path)

    text = ""

    for sheet in excel_file.sheet_names:

        df = excel_file.parse(sheet)

        text += df.astype(str).to_string()

    return text

In [ ]:
# =====================================================
# 8. PPTX 텍스트 추출
# =====================================================

def extract_pptx_text(path):

    prs = Presentation(path)

    text = ""

    for slide in prs.slides:

        for shape in slide.shapes:

            if hasattr(shape, "text"):

                text += shape.text + "\n"

    return text

In [ ]:
# =====================================================
# 9. 통합 Text Extractor
# =====================================================

def extract_text(file_path):

    ext = os.path.splitext(
        file_path
    )[1].lower()


    # PDF
    if ext == ".pdf":

        return extract_pdf_text(file_path)


    # 이미지
    elif ext in [
        ".png",
        ".jpg",
        ".jpeg"
    ]:

        return extract_ocr(file_path)


    # DOCX
    elif ext == ".docx":

        return extract_docx_text(file_path)


    # XLSX
    elif ext == ".xlsx":

        return extract_xlsx_text(file_path)


    # PPTX
    elif ext == ".pptx":

        return extract_pptx_text(file_path)


    # 지원 안 하는 파일
    else:

        return ""

In [ ]:
# =====================================================
# 10. 키워드 추출
# =====================================================

def extract_keywords(text):

    result = kiwi.analyze(text)[0][0]

    keywords = []

    for token in result:

        if token.tag in ['NNG', 'NNP']:

            keywords.append(token.form)

    return keywords

In [ ]:
# =====================================================
# 11. 카테고리 규칙
# =====================================================

CATEGORY_RULES = {

    "세금계산서": [
        "공급가액",
        "부가세",
        "전자세금계산서"
    ],

    "회의록": [
        "회의",
        "참석자",
        "논의사항"
    ],

    "사업계획서": [
        "사업",
        "계획",
        "예산",
        "추진"
    ],

    "계약서": [
        "계약",
        "계약금",
        "체결"
    ],

    "제안서": [
        "제안",
        "솔루션",
        "구축"
    ]
}

In [ ]:
# =====================================================
# 12. Rule 기반 점수 계산
# =====================================================

def calculate_rule_score(keywords):

    scores = {}

    for category, rules in CATEGORY_RULES.items():

        score = 0

        for rule in rules:

            if rule in keywords:
                score += 1

        scores[category] = score

    return scores

In [ ]:
# =====================================================
# 13. 후보 카테고리 추출
# =====================================================

def get_candidate_categories(
    scores,
    top_k=3
):

    sorted_categories = sorted(
        scores,
        key=scores.get,
        reverse=True
    )

    return sorted_categories[:top_k]

In [ ]:
# =====================================================
# 14. Prompt 생성
# =====================================================

def build_prompt(
    filename,
    text,
    candidate_categories
):

    prompt = f"""
너는 기업 문서 자동 분류 시스템이다.

파일명:
{filename}

문서 내용:
{text[:2000]}

후보 카테고리:
{', '.join(candidate_categories)}

반드시 아래 JSON만 출력:

{{
    "category": "",
    "llm_score": 0.0,
    "reason": ""
}}
"""

    return prompt

In [ ]:
# =====================================================
# 15. Rule Score 정규화
# =====================================================

def normalize_rule_score(
    score,
    max_score=5
):

    return round(
        score / max_score,
        2
    )

In [ ]:
# =====================================================
# 16. 최종 Confidence 계산
# =====================================================

def final_confidence(
    rule_score,
    llm_score
):

    confidence = (
        (rule_score * 0.3)
        + (llm_score * 0.7)
    )

    return round(confidence, 2)

In [ ]:
# =====================================================
# 17. 상태 결정
# =====================================================

def determine_status(confidence):

    if confidence >= 0.85:

        return "AUTO"

    elif confidence >= 0.70:

        return "REVIEW"

    else:

        return "MANUAL_QUEUE"

In [ ]:
# =====================================================
# 18. 파일 저장
# =====================================================

def move_file_to_category(
    file_path,
    category
):

    base_folder = "/content/classified"

    category_folder = os.path.join(
        base_folder,
        category
    )

    os.makedirs(
        category_folder,
        exist_ok=True
    )

    filename = os.path.basename(
        file_path
    )

    destination = os.path.join(
        category_folder,
        filename
    )

    shutil.copy(
        file_path,
        destination
    )

    return destination

In [ ]:
# =====================================================
# 19. 지원 파일 형식
# =====================================================

SUPPORTED_EXTENSIONS = [

    ".pdf",

    ".png",
    ".jpg",
    ".jpeg",

    ".docx",

    ".xlsx",

    ".pptx"
]

In [ ]:
# =====================================================
# 20. 전체 파일 수집
# =====================================================

all_files = []

for root, dirs, files in os.walk("/content/data"):

    for file in files:

        if any(
            file.lower().endswith(ext)
            for ext in SUPPORTED_EXTENSIONS
        ):

            full_path = os.path.join(
                root,
                file
            )

            all_files.append(full_path)

print("수집 파일 개수:",
      len(all_files))

print(all_files[:10])

수집 파일 개수: 104
['/content/data/사업계획서/[붙임1] 참여기업 모집공고.pdf', '/content/data/사업계획서/4_(1-2)사업계획서_작성_예시_사업계획서_Part2.pdf', '/content/data/사업계획서/3. 2026년 위탁사업_한국물기술인증원.pdf', '/content/data/사업계획서/사업설명서(국내운송사업).pdf', '/content/data/사업계획서/정책민생경제 성장과 건강보험 보장성.pdf', '/content/data/사업계획서/결재문서본문.pdf', '/content/data/사업계획서/1. 맞춤형 국가장학사업(일반회계).pdf', '/content/data/사업계획서/2026회계연도 사업계획서(진료).pdf', '/content/data/사업계획서/사업계획서 작성서식 (1).pdf', '/content/data/사업계획서/2024년_사업_계획(안).pdf']


In [ ]:
# =====================================================
# 21. 전체 멀티포맷 Pipeline 실행
# =====================================================

import shutil
import os


# =====================================================
# 파일 저장 함수
# =====================================================

def move_file_to_category(
    file_path,
    category
):

    base_folder = "/content/classified"

    category_folder = os.path.join(
        base_folder,
        category
    )

    os.makedirs(
        category_folder,
        exist_ok=True
    )

    filename = os.path.basename(
        file_path
    )

    destination = os.path.join(
        category_folder,
        filename
    )

    shutil.copy(
        file_path,
        destination
    )

    return destination


# =====================================================
# 전체 파일 반복 처리
# =====================================================

for sample_path in all_files:

    try:

        filename = os.path.basename(
            sample_path
        )

        print("\n====================")
        print("처리 파일:", filename)
        print("====================")


        # ---------------------------------
        # 1. 파일 타입별 텍스트 추출
        # ---------------------------------

        text = extract_text(sample_path)

        print("텍스트 길이:",
              len(text))


        # 텍스트 부족 시 skip
        if len(text.strip()) < 20:

            print("텍스트 부족 → SKIP")

            continue


        # ---------------------------------
        # 2. 키워드 추출
        # ---------------------------------

        keywords = extract_keywords(text)

        print("\n키워드:")
        print(keywords[:20])


        # ---------------------------------
        # 3. Rule 기반 점수 계산
        # ---------------------------------

        scores = calculate_rule_score(
            keywords
        )

        print("\n카테고리별 Rule Score")

        for category, score in scores.items():

            print(category, ":", score)


        # ---------------------------------
        # 4. 후보 카테고리 생성
        # ---------------------------------

        candidate_categories = (
            get_candidate_categories(scores)
        )

        print("\n후보 카테고리:")
        print(candidate_categories)


        # ---------------------------------
        # 5. Prompt 생성
        # ---------------------------------

        prompt = build_prompt(
            filename,
            text,
            candidate_categories
        )


        # =================================================
        # Mock LLM Score 생성
        # =================================================

        mock_llm_score = round(

            min(
                0.6 + (
                    max(scores.values()) * 0.08
                ),
                0.99
            ),

            2
        )


        # =================================================
        # Mock LLM 결과
        # =================================================

        llm_result = {

            "category":
                candidate_categories[0],

            "llm_score":
                mock_llm_score,

            "reason":
                "키워드 및 문맥 기반 분류"
        }


        # ---------------------------------
        # 6. Rule Score 정규화
        # ---------------------------------

        rule_score = normalize_rule_score(
            max(scores.values())
        )

        print("\n정규화된 Rule Score:")
        print(rule_score)


        # ---------------------------------
        # 7. LLM Score
        # ---------------------------------

        llm_score = llm_result["llm_score"]

        print("\nLLM Score:")
        print(llm_score)


        # ---------------------------------
        # 8. Final Confidence
        # ---------------------------------

        confidence = final_confidence(
            rule_score,
            llm_score
        )

        print("\n최종 Confidence:")
        print(confidence)


        # ---------------------------------
        # 9. 상태 결정
        # ---------------------------------

        status = determine_status(
            confidence
        )

        print("\n상태:")
        print(status)


        # ---------------------------------
        # 10. 파일 저장
        # ---------------------------------

        saved_path = move_file_to_category(
            sample_path,
            llm_result["category"]
        )

        print("\n저장 위치:")
        print(saved_path)


        # ---------------------------------
        # 최종 결과
        # ---------------------------------

        print("\n최종 카테고리:")
        print(llm_result["category"])


    except Exception as e:

        print("\n오류 발생 파일:")
        print(sample_path)

        print("\n에러 내용:")
        print(e)


처리 파일: [붙임1] 참여기업 모집공고.pdf
텍스트 길이: 2195

키워드:
['대전', '정보', '문화', '산업', '진흥원', '공고', '대전방산혁신클러스터', '드론', '특화', '국방', '신규', '진입', '창업', '지원', '대전', '방산', '진입', '드론', '창업', '기업']

카테고리별 Rule Score
세금계산서 : 0
회의록 : 0
사업계획서 : 2
계약서 : 1
제안서 : 1

후보 카테고리:
['사업계획서', '계약서', '제안서']

정규화된 Rule Score:
0.4

LLM Score:
0.76

최종 Confidence:
0.65

상태:
MANUAL_QUEUE

저장 위치:
/content/classified/사업계획서/[붙임1] 참여기업 모집공고.pdf

최종 카테고리:
사업계획서

처리 파일: 4_(1-2)사업계획서_작성_예시_사업계획서_Part2.pdf
텍스트 길이: 2325

키워드:
['사업', '계획서', '작성', '예시', '사업', '계획서', '중소기업', '사업', '계획서', '작성', '도움', '사업', '계획서', '의', '항목', '작성', '주요', '내용', '작성', '예시']

카테고리별 Rule Score
세금계산서 : 0
회의록 : 0
사업계획서 : 2
계약서 : 0
제안서 : 1

후보 카테고리:
['사업계획서', '제안서', '세금계산서']

정규화된 Rule Score:
0.4

LLM Score:
0.76

최종 Confidence:
0.65

상태:
MANUAL_QUEUE

저장 위치:
/content/classified/사업계획서/4_(1-2)사업계획서_작성_예시_사업계획서_Part2.pdf

최종 카테고리:
사업계획서

처리 파일: 3. 2026년 위탁사업_한국물기술인증원.pdf
텍스트 길이: 872

키워드:
['위탁', '사업', '연도', '예', '결산', '액', '백만원', '결산', '결산', '결산', '결산', '결산', '예산',